# LightGBM baseline — context and objective

This notebook prepares the team's first reproducible ML experiment. Person 1 has supplied cleaned transaction data; Person 2 is developing graph features, and Person 4 is developing behavioural features. We establish a **baseline** now so that we can measure the contribution of those new features when they arrive.

The current score is a starting reference, not the final model or submission. More features may help, but improvement must be demonstrated using the same internal split and evaluation metric. This notebook currently trains **LightGBM only**; Random Forest comparison belongs to a later experiment.

## Which data do we use?

| Dataset | Rows | Labels available? | Use in this notebook |
|---|---:|---|---|
| Cleaned train | 40,131 | Yes: `is_suspicious` | Split internally for training and evaluation |
| Official validation | 33,862 | No | Not read here |
| Official test | 40,101 | No | Reserved for a later final prediction stage |

The training label is `0` for legitimate and `1` for suspicious. There are 1,250 suspicious examples (about 3.11%), so predicting every transaction as legitimate would produce high accuracy while detecting no suspicious transactions. For that reason, **Average Precision (AP)** is our primary metric.

## How to run

Select the project Python environment and run all cells from top to bottom in a **new kernel**. Alternatively, run `.venv/bin/python notebooks/run_ml_baseline.py` from the repository root. Dependency versions are recorded in `requirements-ml.txt`.

The next cell loads only cleaned training data, checks hashes and labels, and prints the dataset shape, class balance and environment versions.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "notebooks/02_ml_baseline.ipynb").is_file()
             and (p / "data/processed/train_cleaned.parquet").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run from the repository root or a directory inside it")

DATA_DIR = ROOT / "data" / "processed"
train = pd.read_parquet(DATA_DIR / "train_cleaned.parquet")

print("Shape:", train.shape)
print("\nClass distribution:")
print(train["is_suspicious"].value_counts())

print("\nPercentages:")
print((train["is_suspicious"].value_counts(normalize=True) * 100).round(2))

import sys
import json
import hashlib
import importlib.metadata as metadata
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, average_precision_score, roc_auc_score,
                             precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report)

if train.TRANSACTION_HASH.isna().any() or not train.TRANSACTION_HASH.is_unique:
    raise ValueError("Training transaction hashes must be nonmissing and unique")
if train.is_suspicious.isna().any() or not train.is_suspicious.isin([0, 1]).all():
    raise ValueError("Training targets must be complete and binary")
if set(train.is_suspicious.unique()) != {0, 1}:
    raise ValueError("Both target classes are required")
print("Python:", sys.version, sys.executable)
versions = {p: metadata.version(p) for p in ["pandas", "numpy", "pyarrow", "lightgbm", "scikit-learn", "nbformat", "nbclient", "ipykernel"]}
print("Versions:", versions)

# Import the feature adapter when running from either root or notebooks/.
sys.path.insert(0, str(ROOT / "notebooks"))


Shape: (40131, 36)

Class distribution:
is_suspicious
0    38881
1     1250
Name: count, dtype: int64

Percentages:
is_suspicious
0    96.89
1     3.11
Name: proportion, dtype: float64
Python: 3.13.2 (v3.13.2:4f8bb3947cf, Feb  4 2025, 11:51:10) [Clang 15.0.0 (clang-1500.3.9.4)] /Users/marinescucatinca/Hackaton/decode-detect-hackathon/.venv/bin/python
Versions: {'pandas': '3.1.0', 'numpy': '2.5.4', 'pyarrow': '26.0.0', 'lightgbm': '4.7.0', 'scikit-learn': '1.9.1', 'nbformat': '5.11.1', 'nbclient': '0.11.0', 'ipykernel': '7.4.0'}


## Receive graph and behavioural features

Person 2's graph features describe wallet relationships and network patterns, such as fan-in, fan-out or connected groups. Person 4's behavioural features describe activity over time, such as transaction frequency, recent volumes or unusual amounts. These may contain useful signals that the initial cleaning features do not capture.

The configuration below lets us compare four LightGBM experiments:

| Graph path | Behaviour path | Experiment |
|---|---|---|
| `None` | `None` | Cleaning baseline |
| File path | `None` | Baseline + graph |
| `None` | File path | Baseline + behaviour |
| File path | File path | Baseline + both |

Leave both paths as `None` while waiting for the real files. When a file arrives, set its path and rerun the **entire notebook in a new kernel**. No placeholder features are invented. Each experiment has a separate output directory so that the baseline remains available for comparison.

Each supplied file must contain exactly one row for every training `TRANSACTION_HASH`, with numeric or boolean feature columns and no labels or annotation metadata. We join by hash, not row position, and reject missing/extra hashes, duplicates and invalid values. Intentional missing numeric values are allowed only for explicitly configured groups; infinities are always rejected. Prefixes distinguish graph and behavioural columns. These checks cannot prove that a feature was computed without labels: confirm feature definitions and history/time-window assumptions with each teammate. Full handoff details are in `docs/feature_handoff.md`.

**Current experiment:** Thanmaya’s six behaviour features (commit `1935686`) are configured below. Missing first-payment gaps are preserved explicitly and handled natively by LightGBM. Set behaviour to `None` to rerun the cleaning-only baseline. No graph features are added in this experiment.

In [2]:
# Change only these paths when the teammates deliver their TRAIN features.
# None means the group has not arrived yet; no placeholder features are invented.
FEATURE_FILES = {
    "graph": None,      # ROOT / "data/processed/train_graph_features.csv"
    "behaviour": ROOT / "outputs/thanmaya_behaviour/train_cleaned_features.parquet",
}
from ml_feature_inputs import merge_team_features
# Thanmaya defines unknown first-payment gaps as missing, not zero.
# LightGBM supports these missing numeric values natively; no imputer is fitted.
ALLOW_MISSING_FEATURE_GROUPS = {"behaviour"}
train, team_feature_columns, team_sources = merge_team_features(
    train, FEATURE_FILES, allow_missing_groups=ALLOW_MISSING_FEATURE_GROUPS,
)
feature_groups = [name for name in ("graph", "behaviour") if FEATURE_FILES[name] is not None]
EXPERIMENT_NAME = "baseline" if not feature_groups else "baseline_" + "_".join(feature_groups)
print("Experiment:", EXPERIMENT_NAME)
print("Team features added:", len(team_feature_columns))
print("Sources:", team_sources)


Experiment: baseline_behaviour
Team features added: 6
Sources: {'behaviour': '/Users/marinescucatinca/Hackaton/decode-detect-hackathon/outputs/thanmaya_behaviour/train_cleaned_features.parquet'}


## Prepare features and keep the same 80/20 split

`X` contains the information the model may use; `y` contains the `is_suspicious` labels it must learn to predict. We use an explicit list of 18 baseline features, then append only the validated teammate features selected above. Hashes, account identifiers, raw JSON, annotation metadata and raw datetime columns are excluded from model inputs.

The internal split uses `test_size=0.20`, `random_state=42` and `stratify=y`:

- **32,104 transactions** form the training subset, with 1,000 suspicious examples.
- **8,027 transactions** form the internal holdout, with 250 suspicious examples.

Stratification keeps approximately the same class balance in both subsets. The fixed seed and unchanged input row order preserve the same hash assignments across feature experiments. We save those assignments so other team models can use them too. This internal holdout is separate from the organiser's unlabelled validation dataset.

Categorical levels are learned from internal training rows only. Calendar and payment features come from cleaning; `delivered_usd` values the amount received by the recipient. Constant baseline columns are reported and retained for consistency. Raw nominal amounts and reference prices remain initial baseline inputs, whose usefulness can be reviewed later.

A random split can share accounts and time periods between training and holdout. It is our agreed initial comparison setup; a later temporal/group audit can assess how much the score depends on those relationships.

In [3]:
# Explicit feature allowlist prevents accidental inclusion of label metadata or IDs.
# These are the 18 baseline features agreed in the team's initial preparation.
BASE_FEATURE_COLUMNS = [
    "TIMESTAMP", "TRANSACTION_TYPE", "SOURCE_FEE", "FEE", "FLAGS",
    "transfer_count", "unique_senders", "unique_receivers", "unique_tokens",
    "hour", "day_of_week", "is_weekend", "delivered_value", "is_xrp_payment",
    "has_sendmax", "has_paths", "usd_price", "delivered_usd",
]
FEATURE_COLUMNS = BASE_FEATURE_COLUMNS + team_feature_columns
X = train[FEATURE_COLUMNS].copy()
y = train["is_suspicious"].astype(int)

# Split row indices first so hashes, features and targets remain aligned.
train_idx, val_idx = train_test_split(
    train.index.to_numpy(), test_size=0.20, random_state=42, stratify=y,
)
X_train, X_val = X.loc[train_idx].copy(), X.loc[val_idx].copy()
y_train, y_val = y.loc[train_idx].copy(), y.loc[val_idx].copy()
if set(train_idx).intersection(val_idx) or set(train_idx).union(val_idx) != set(train.index):
    raise ValueError("Internal split must be disjoint and cover all training rows")

# Learn categorical vocabulary only from the internal training subset.
category_levels = {}
for col in ["TRANSACTION_TYPE"]:
    levels = sorted(X_train[col].dropna().unique().tolist())
    category_levels[col] = levels
    dtype = pd.CategoricalDtype(categories=levels, ordered=False)
    X_train[col] = X_train[col].astype(dtype)
    X_val[col] = X_val[col].astype(dtype)

numeric_columns = [c for c in FEATURE_COLUMNS if c not in category_levels]
for name, frame in [("train", X_train), ("holdout", X_val)]:
    # Preserve intentional NaN only in explicitly permitted teammate groups.
    required_finite = [c for c in numeric_columns
                       if not any(c.startswith(group + "__") for group in ALLOW_MISSING_FEATURE_GROUPS)]
    if (np.isinf(frame[numeric_columns].to_numpy(dtype=float)).any()
            or not np.isfinite(frame[required_finite].to_numpy(dtype=float)).all()):
        raise ValueError(f"{name}: nonfinite numeric features")
    print(f"{name}: {frame.shape}, suspicious={int(y.loc[frame.index].sum())}, missing={int(frame.isna().sum().sum())}")
print("Features:", FEATURE_COLUMNS)
print("Categorical levels (training only):", category_levels)
print("Constant training features:", X_train.columns[X_train.nunique(dropna=False) <= 1].tolist())


train: (32104, 24), suspicious=1000, missing=529
holdout: (8027, 24), suspicious=250, missing=149
Features: ['TIMESTAMP', 'TRANSACTION_TYPE', 'SOURCE_FEE', 'FEE', 'FLAGS', 'transfer_count', 'unique_senders', 'unique_receivers', 'unique_tokens', 'hour', 'day_of_week', 'is_weekend', 'delivered_value', 'is_xrp_payment', 'has_sendmax', 'has_paths', 'usd_price', 'delivered_usd', 'behaviour__sender_tx_count_1h', 'behaviour__sender_tx_count_24h', 'behaviour__seconds_since_previous_tx', 'behaviour__sender_unique_recipients_24h', 'behaviour__sender_outflow_usd_24h', 'behaviour__sender_inflow_usd_24h']
Categorical levels (training only): {'TRANSACTION_TYPE': ['Payment']}
Constant training features: ['TRANSACTION_TYPE', 'FLAGS']


## Train LightGBM with the team's starting parameters

LightGBM builds decision trees sequentially, with each new tree improving the current predictions. We keep the team's initial `B_PARAMS` unchanged for fair feature comparisons:

| Parameter | Starting value | Meaning |
|---|---:|---|
| `n_estimators` | 700 | Maximum number of boosting iterations |
| `learning_rate` | 0.02 | Contribution of each additional tree |
| `num_leaves` | 63 | Maximum leaves per tree |
| `max_depth` | 8 | Maximum tree depth |
| `min_child_samples` | 150 | Minimum observations in a leaf |
| `colsample_bytree` | 0.8 | Fraction of features considered for each tree |
| `reg_lambda` | 1.0 | L2 regularisation |
| `random_state` | 0 | Model random seed |

No hyperparameter search, resampling or class weighting is introduced. Operational settings use four CPU threads and request deterministic histogram construction.

We monitor **Average Precision only** on the internal holdout. Early stopping has a patience of 50 rounds: training ends if AP stops improving for that long, or when the 700-round limit is reached. Predictions use the best observed iteration, which can be earlier than the last trained iteration. This is why 700 is a limit rather than a promise to use all trees in the saved model.

In [4]:
# Team-provided baseline parameters: no hyperparameter search.
B_PARAMS = {
    "n_estimators": 700,
    "learning_rate": 0.02,
    "num_leaves": 63,
    "max_depth": 8,
    "min_child_samples": 150,
    "objective": "binary",
    "class_weight": None,
    "colsample_bytree": 0.8,
    "reg_lambda": 1.0,
    "random_state": 0,
    "verbosity": -1,
}
# Operational settings: evaluate only AP; fixed CPU parallelism and deterministic histograms.
RUN_PARAMS = {"metric": "average_precision", "n_jobs": 4,
              "deterministic": True, "force_col_wise": True}
model = lgb.LGBMClassifier(**B_PARAMS, **RUN_PARAMS)
model.fit(
    X_train, y_train, eval_X=X_val, eval_y=y_val, eval_names=["holdout"],
    eval_metric="average_precision",
    callbacks=[lgb.early_stopping(50, first_metric_only=True, verbose=False)],
)
y_prob = model.predict_proba(X_val, num_iteration=model.best_iteration_)[:, 1]
if len(y_prob) != len(y_val) or not np.isfinite(y_prob).all() or ((y_prob < 0) | (y_prob > 1)).any():
    raise ValueError("Invalid holdout scores")
ap = float(average_precision_score(y_val, y_prob))
# Verify the early-stopping metric agrees with the organiser's sklearn AP definition.
np.testing.assert_allclose(ap, model.best_score_["holdout"]["average_precision"], atol=1e-10)
print("Best iteration:", model.best_iteration_, "/", B_PARAMS["n_estimators"])


Best iteration: 699 / 700


## Training vs internal test accuracy, then ranking metrics

`predict_proba()` returns a continuous suspicion score between 0 and 1. The competition ranks transactions by these scores, so we use probabilities rather than binary predictions.

- **Average Precision (AP):** the primary ranking metric. Higher is better; the organiser's score is `100 × AP`. AP is not the percentage of suspicious transactions detected, and it differs from trapezoidal integration of a precision-recall curve.
- **Precision at 0.5:** among transactions flagged at the fixed threshold, how many are suspicious?
- **Recall at 0.5:** among all suspicious transactions, how many are flagged?
- **F1 at 0.5:** a combined measure of precision and recall.
- **Top 1% diagnostics:** precision and recall when reviewing only the highest-scoring 1% of holdout transactions.

The threshold of 0.5 is fixed in advance, not tuned in this experiment. A model can rank transactions reasonably while having low recall at that threshold; AP and recall describe different aspects of performance. ROC AUC is reported as an additional diagnostic. The confusion matrix gives counts of correct decisions, false alarms and missed suspicious transactions.

Feature importance reports how much features contributed to fitted tree splits. It helps inspect the model but does not establish causation or prove the absence of leakage.

The next cell starts with an **ACCURACY** table for training (80%) and internal test/holdout (20%), at the same fixed 0.5 threshold. These values are calculated from the current fitted model, not hard-coded. The official unlabelled test is not evaluated here. An always-normal reference is shown alongside because legitimate transactions make up about 96.89% of the dataset; high accuracy alone can hide missed suspicious transactions. Use AP for the competition ranking and recall to understand detection.


In [5]:
# Threshold is fixed in advance, not tuned on this holdout.
THRESHOLD = 0.5
y_pred = (y_prob >= THRESHOLD).astype(int)

# Show accuracy explicitly on both subsets using the SAME selected model and threshold.
train_prob = model.predict_proba(X_train, num_iteration=model.best_iteration_)[:, 1]
train_pred = (train_prob >= THRESHOLD).astype(int)
training_accuracy = float(accuracy_score(y_train, train_pred))
holdout_accuracy = float(accuracy_score(y_val, y_pred))
accuracy_summary = pd.DataFrame({
    "Dataset": ["Training (80%)", "Internal test / holdout (20%)"],
    "Transactions": [len(y_train), len(y_val)],
    "Correct predictions": [int((train_pred == y_train.to_numpy()).sum()),
                            int((y_pred == y_val.to_numpy()).sum())],
    "Accuracy (%)": [100 * training_accuracy, 100 * holdout_accuracy],
    "Always-normal reference (%)": [100 * float((y_train == 0).mean()),
                                    100 * float((y_val == 0).mean())],
})
print("\nACCURACY — threshold = 0.5 (official test not used)")
print(accuracy_summary.to_string(index=False, float_format=lambda value: f"{value:.2f}"))
print(f"Training accuracy: {training_accuracy:.2%}")
print(f"Internal test accuracy: {holdout_accuracy:.2%}")
print("Accuracy counts all correct decisions. AP remains the official competition metric.")

holdout = pd.DataFrame({
    "TRANSACTION_HASH": train.loc[val_idx, "TRANSACTION_HASH"].to_numpy(),
    "is_suspicious": y_val.to_numpy(), "score": y_prob,
})
if not holdout.TRANSACTION_HASH.is_unique or set(holdout.TRANSACTION_HASH) != set(train.loc[val_idx, "TRANSACTION_HASH"]):
    raise ValueError("Holdout prediction hash coverage differs")
# Diagnostic top 1% queue, ties broken deterministically by hash ascending.
queue_size = max(1, int(np.ceil(0.01 * len(holdout))))
queue = holdout.sort_values(["score", "TRANSACTION_HASH"], ascending=[False, True]).head(queue_size)
metrics = {
    "training_accuracy_at_0_5": training_accuracy,
    "holdout_accuracy_at_0_5": holdout_accuracy,
    "average_precision": ap,
    "reference_100_times_ap": 100 * ap,
    "random_ranking_reference_ap": float(y_val.mean()),
    "roc_auc": float(roc_auc_score(y_val, y_prob)),
    "threshold": THRESHOLD,
    "precision_at_0_5": float(precision_score(y_val, y_pred, zero_division=0)),
    "recall_at_0_5": float(recall_score(y_val, y_pred, zero_division=0)),
    "f1_at_0_5": float(f1_score(y_val, y_pred, zero_division=0)),
    "top_1_percent_count": queue_size,
    "top_1_percent_precision": float(queue.is_suspicious.mean()),
    "top_1_percent_recall": float(queue.is_suspicious.sum() / y_val.sum()),
    "confusion_matrix_tn_fp_fn_tp": confusion_matrix(y_val, y_pred, labels=[0, 1]).ravel().tolist(),
    "training_rows": len(X_train), "holdout_rows": len(X_val),
    "training_suspicious": int(y_train.sum()), "holdout_suspicious": int(y_val.sum()),
    "feature_count": len(FEATURE_COLUMNS), "best_iteration": int(model.best_iteration_),
    "trained_iterations": len(model.evals_result_["holdout"]["average_precision"]),
}
print(json.dumps(metrics, indent=2))
print(classification_report(y_val, y_pred, labels=[0, 1], digits=4, zero_division=0))
importance = pd.DataFrame({"feature": FEATURE_COLUMNS,
                           "gain": model.booster_.feature_importance(importance_type="gain")}).sort_values("gain", ascending=False)
print("Feature importance (gain; descriptive, not causal):")
print(importance.to_string(index=False))



ACCURACY — threshold = 0.5 (official test not used)
                      Dataset  Transactions  Correct predictions  Accuracy (%)  Always-normal reference (%)
               Training (80%)         32104                31379         97.74                        96.89
Internal test / holdout (20%)          8027                 7809         97.28                        96.89
Training accuracy: 97.74%
Internal test accuracy: 97.28%
Accuracy counts all correct decisions. AP remains the official competition metric.
{
  "training_accuracy_at_0_5": 0.9774171442810865,
  "holdout_accuracy_at_0_5": 0.9728416593995266,
  "average_precision": 0.41718847712038615,
  "reference_100_times_ap": 41.71884771203862,
  "random_ranking_reference_ap": 0.031144886009717203,
  "roc_auc": 0.8639372508679439,
  "threshold": 0.5,
  "precision_at_0_5": 0.9210526315789473,
  "recall_at_0_5": 0.14,
  "f1_at_0_5": 0.24305555555555555,
  "top_1_percent_count": 81,
  "top_1_percent_precision": 0.691358024691358,
  "

## Save the experiment and prepare the next comparison

The notebook saves metrics, feature importance, holdout probabilities, the hash split manifest and a model trained on the internal 80% subset. All generated artifacts go under ignored `outputs/`, and the code verifies that the saved model reproduces its predictions. These are experiment artifacts, not a competition submission.

When real graph/behaviour files are configured, their paths and checksums are recorded and the results go into a separate experiment directory. If a baseline report exists with the same cleaned input, split and model settings, the notebook also prints and saves the change in AP. A positive change is evidence of improvement on this development split; it does not guarantee the same gain on the official test.

The internal holdout also chooses the boosting iteration through early stopping. Its score is therefore a **development estimate**, not an independent final test score. Avoid repeatedly choosing features or thresholds on it and then treating the result as unbiased.

After the team compares models and features, a separate final stage can refit the chosen configuration on all labelled training rows and produce one continuous score per official test transaction. This notebook stops at internal evaluation and does not overwrite `src/predictions.csv`.

In [6]:
# Local experiment artifacts only, all inside ignored outputs/.
OUTPUT_DIR = ROOT / "outputs" / ("ml_baseline" if EXPERIMENT_NAME == "baseline" else f"ml_{EXPERIMENT_NAME}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
accuracy_summary.to_csv(OUTPUT_DIR / "accuracy_summary.csv", index=False)
split_manifest = train[["TRANSACTION_HASH"]].copy()
split_manifest["partition"] = "internal_train"
split_manifest.loc[val_idx, "partition"] = "internal_holdout"
split_manifest.to_csv(OUTPUT_DIR / "split_manifest.csv", index=False)
holdout.to_csv(OUTPUT_DIR / "holdout_predictions.csv", index=False)
importance.to_csv(OUTPUT_DIR / "feature_importance.csv", index=False)
model.booster_.save_model(str(OUTPUT_DIR / "internal_train_model.txt"), num_iteration=model.best_iteration_)
report = {
    "experiment": EXPERIMENT_NAME, "team_sources": team_sources,
    "allow_missing_groups": sorted(ALLOW_MISSING_FEATURE_GROUPS),
    "team_source_sha256": {name: hashlib.sha256(Path(path).read_bytes()).hexdigest() for name, path in team_sources.items()},
    "metrics": metrics, "team_params": B_PARAMS, "run_params": RUN_PARAMS,
    "split": {"test_size": 0.20, "random_state": 42, "stratify": "is_suspicious"},
    "feature_columns": FEATURE_COLUMNS, "category_levels": category_levels,
    "versions": versions, "python": sys.version, "working_directory": str(Path.cwd()),
    "input": "data/processed/train_cleaned.parquet",
    "input_sha256": hashlib.sha256((DATA_DIR / "train_cleaned.parquet").read_bytes()).hexdigest(),
    "scope": "Internal development baseline; holdout also selects boosting iteration. No official validation/test data read. Not a final submission model.",
}
(OUTPUT_DIR / "metrics.json").write_text(json.dumps(report, indent=2) + "\n")
# Check saved artifacts, row coverage, score integrity and model reload.
saved = pd.read_csv(OUTPUT_DIR / "holdout_predictions.csv")
pd.testing.assert_frame_equal(saved, holdout, check_exact=False, rtol=1e-12, atol=1e-15)
manifest = pd.read_csv(OUTPUT_DIR / "split_manifest.csv")
if len(manifest) != len(train) or not manifest.TRANSACTION_HASH.is_unique or set(manifest.TRANSACTION_HASH) != set(train.TRANSACTION_HASH):
    raise ValueError("Split manifest input/output hash coverage differs")
restored = lgb.Booster(model_file=str(OUTPUT_DIR / "internal_train_model.txt"))
np.testing.assert_allclose(restored.predict(X_val, num_threads=4), y_prob, rtol=1e-12, atol=1e-15)
print("PASS: split coverage, score bounds, native AP agreement, artifact round trips and model reload.")
print("Artifacts:", OUTPUT_DIR)


# Compare only against the same cleaned input, split and model settings.
baseline_path = ROOT / "outputs/ml_baseline/metrics.json"
if EXPERIMENT_NAME != "baseline" and baseline_path.is_file():
    baseline = json.loads(baseline_path.read_text())
    if (baseline["input_sha256"] == report["input_sha256"]
            and baseline["split"] == report["split"]
            and baseline["team_params"] == B_PARAMS and baseline["run_params"] == RUN_PARAMS):
        comparison = pd.DataFrame([
            {"experiment": "baseline", "feature_count": baseline["metrics"]["feature_count"],
             "average_precision": baseline["metrics"]["average_precision"]},
            {"experiment": EXPERIMENT_NAME, "feature_count": metrics["feature_count"],
             "average_precision": ap},
        ])
        comparison.to_csv(OUTPUT_DIR / "comparison_to_baseline.csv", index=False)
        print(comparison.to_string(index=False))
        print("AP change vs baseline:", ap - baseline["metrics"]["average_precision"])
    else:
        print("Baseline comparison skipped: input, split or parameters differ; rerun the baseline.")


PASS: split coverage, score bounds, native AP agreement, artifact round trips and model reload.
Artifacts: /Users/marinescucatinca/Hackaton/decode-detect-hackathon/outputs/ml_baseline_behaviour
        experiment  feature_count  average_precision
          baseline             18           0.341644
baseline_behaviour             24           0.417188
AP change vs baseline: 0.07554476566533991


## Recorded experiment 

With the same seed-42 split and original team parameters, adding Thanmaya's six behaviour features increased internal **Average Precision from 0.341644 to 0.417188**. At the fixed threshold 0.5, detected suspicious transactions increased from 23/250 to 35/250, with false positives decreasing from 6 to 3. This is a development comparison, not an official test score.

Source: Person 4 branch `behaviour-features-pr-04`, commit `1935686`. Generate the train-only feature table first with `.venv/bin/python notebooks/run_behaviour_features.py`. Then run this notebook; the behaviour path is configured above. The model preserves meaningful missing gaps for first observed transactions. Graph features remain disabled.

See `docs/thanmaya_behaviour_comparison.md` for full metrics, source adaptations, checks and limitations. Future runs may produce different results if feature files or settings change; consult their saved metrics.